# Exploración y verificación
# Banco: transacciones


In [41]:
import pandas as pd
import duckdb

df = pd.read_csv("datos/transacciones.csv", sep=";")
df.shape


(2575, 10)

In [42]:
df.head()

,id_transaccion,fecha_hora,id_cuenta,tipo,monto,moneda,canal,id_sucursal,sucursal,estado
0,TX000001,2024-07-28 02:23:28,10238,pago_servicio,436922.54,ARS,sucursal,S03,Belgrano,aprobada
1,TX000002,2025-01-03 17:29:09,10215,pago_servicio,"87.484,74",ARS,app,S03,Belgrano,aprobada
2,TX000003,2025-07-04 04:36:29,10035,deposito,637694.71,ARS,web,S03,Belgrano,aprobada
3,TX000004,2024-05-20 02:33:16,10033,deposito,538665.09,ARS,cajero,S04,Caballito,aprobada
4,TX000330,2024-11-25 09:03:27,10237,EXTRACCION,408778.04,ARS,app,S02,Palermo,aprobada


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2575 entries, 0 to 2574
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id_transaccion  2575 non-null   str  
 1   fecha_hora      2575 non-null   str  
 2   id_cuenta       2575 non-null   int64
 3   tipo            2575 non-null   str  
 4   monto           2575 non-null   str  
 5   moneda          2575 non-null   str  
 6   canal           2575 non-null   str  
 7   id_sucursal     2575 non-null   str  
 8   sucursal        2443 non-null   str  
 9   estado          2575 non-null   str  
dtypes: int64(1), str(9)
memory usage: 384.2 KB


In [44]:
df.isna().sum()

id_transaccion      0
fecha_hora          0
id_cuenta           0
tipo                0
monto               0
moneda              0
canal               0
id_sucursal         0
sucursal          132
estado              0
dtype: int64

In [45]:
df["moneda"].value_counts()

moneda
ARS    2049
USD     397
ars     114
usd      15
Name: count, dtype: int64

In [46]:
df["tipo"].value_counts()

tipo
deposito          626
pago_servicio     608
extraccion        594
transferencia     544
EXTRACCION         23
TRANSFERENCIA      20
 PAGO_SERVICIO     20
DEPOSITO           19
EXTRACCION         18
 EXTRACCION        17
PAGO_SERVICIO      16
PAGO_SERVICIO      16
 TRANSFERENCIA     15
DEPOSITO           14
 DEPOSITO          13
TRANSFERENCIA      12
Name: count, dtype: int64

In [47]:
print("Ids repetidos:", df.duplicated(subset=["id_transaccion"]).sum())
print("Filas idénticas repetidas:", df.duplicated().sum())

Ids repetidos: 75
Filas idénticas repetidas: 75


In [48]:
fecha = pd.to_datetime(df["fecha_hora"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
print("Fechas que no cumplen YYYY-mm-dd:", fecha.isna().sum())
df[fecha.isna()]["fecha_hora"].head()

Fechas que no cumplen YYYY-mm-dd: 259


19    06/07/2024 15:00:21
20    01/11/2025 18:41:17
26    02/05/2024 08:32:00
27    10/11/2024 22:58:24
33    13/01/2025 03:27:11
Name: fecha_hora, dtype: str

In [49]:
monto = pd.to_numeric(df["monto"], errors="coerce")
print("Montos que no se pueden convertir a número:", monto.isna().sum())
print("Montos menores o iguales a cero:", len(df[monto <= 0]))
df[monto.isna()]["monto"].head()

Montos que no se pueden convertir a número: 155
Montos menores o iguales a cero: 106


1       87.484,74
16     404.795,87
36     109.967,75
99     281.864,43
108    766.093,73
Name: monto, dtype: str

## Verificación de resultados del pipeline

In [50]:
con = duckdb.connect("pipeline.duckdb", read_only=True)
rechazados = con.execute("SELECT * FROM rechazados").df()
validos = con.execute("SELECT * FROM validos").df()
final = con.execute("SELECT * FROM tabla_final").df()
con.close()

print("Válidos:", len(validos), "- Rechazados:", len(rechazados))
print("Tabla final:", len(final), "- ids repetidos:", final.duplicated(subset=["id_transaccion"]).sum())

Válidos: 1509 - Rechazados: 1066
Tabla final: 2500 - ids repetidos: 0


In [51]:
rechazados["motivo_rechazo"].value_counts()

motivo_rechazo
fecha_hora no cumple el formato YYYY-mm-dd HH:MM:SS    241
tipo no estandarizado                                  197
id_transaccion duplicado                               150
monto con formato numérico no estandarizado            145
moneda no estandarizada                                121
campo con valor nulo                                   118
monto menor o igual a cero                              94
Name: count, dtype: int64

In [52]:
# 1) Monto aprobado en ARS por tipo
final[(final["estado"] == "aprobada") & (final["moneda"] == "ARS")].groupby("tipo")["monto"].sum().astype("int")

tipo
deposito         244315991
extraccion       214557718
pago_servicio    215817670
transferencia    199063880
Name: monto, dtype: int64

In [53]:
# 2) Transacciones por canal
final["canal"].value_counts()

canal
web         646
app         642
cajero      620
sucursal    592
Name: count, dtype: int64

In [54]:
# 3) Cuenta con más movimientos
final["id_cuenta"].value_counts().head(1)

id_cuenta
10092    15
Name: count, dtype: int64

In [55]:
# 4) % de rechazadas por sucursal
final[final["estado"] == "rechazada"]["sucursal"].value_counts() / final["sucursal"].value_counts() * 100

sucursal
Belgrano    7.71
Caballito   8.01
Centro      8.60
La Plata    8.87
Palermo     9.02
Name: count, dtype: float64

In [56]:
# 5) Transacciones por mes (primeros 6 meses: enero a junio 2024)
f = final[(final["fecha_hora"] >= "2024-01-01") & (final["fecha_hora"] < "2024-07-01")]
f["fecha_hora"].dt.month.value_counts()

fecha_hora
5    113
4    107
3    103
1    102
6    101
2     99
Name: count, dtype: int64